# 04b — Assembly Pipeline (v1 Feature Replication)

Assembles the v2 Parquet from 04a's smoothed NDVI features, replicating v1's full feature set at 100 m² tile resolution.

**Sources:**
- `data/spark/ndvi_weekly/` — per-tile Hampel+smooth+interpolate features from 04a
- `data/weather_daily.csv` — daily PRISM weather (single station)
- `data/terrain_100m2.csv` — static terrain features per tile
- `data/soil_100m2.csv` — static SSURGO soil features per tile

**v1 feature replication:**
- NDVI: `{w}_ndvi_smooth_mean/std/slope` (all weeks) + `{w}_ndvi_anomaly` (weeks 36–43)
- Climate weeks 28–43: ppt, tmax, tmin, tmean, vpdmax, vpdmin, cumulative_gdd
- Derived climate: water_availability, diurnal_temp_range, stress_index (v1 naming)
- Terrain: all static features + local_relief + total_relief_log
- Soil: all SSURGO features
- Target: `ndvi_anomaly_harvest` (null-safe mean of weeks 36–43 anomaly)

Output: `data/spark/assembled_v2/`

In [1]:
import os
import numpy as np
import pandas as pd

os.environ.setdefault('JAVA_HOME', '/usr/lib/jvm/java-17-openjdk-amd64')
os.environ['PYSPARK_PYTHON']        = '/home/simonhans/anaconda3/envs/GrapeExpectationsML/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/home/simonhans/anaconda3/envs/GrapeExpectationsML/bin/python'

NDVI_WEEKLY = '../data/spark/ndvi_weekly'
TERRAIN     = '../data/terrain_100m2.csv'
SOIL        = '../data/soil_100m2.csv'
WEATHER     = '../data/weather_daily.csv'
OUT_DIR     = '../data/spark/assembled_v2'

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName('GrapeExpectations-v2-AssemblyV2')
    .master('local[*]')
    .config('spark.driver.memory', '10g')
    .config('spark.driver.maxResultSize', '2g')
    .config('spark.sql.shuffle.partitions', '200')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .getOrCreate())

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

In [3]:
from pyspark.sql import functions as F
from functools import reduce

ndvi = spark.read.parquet(NDVI_WEEKLY)
print(f'NDVI weekly rows: {ndvi.count():,}')
ndvi.show(3)

# Vineyard mean per (year, week) across all tiles — strips vintage, isolates spatial signal
vineyard_mean = (ndvi
    .groupBy('year', 'week')
    .agg(F.mean('ndvi_smooth_mean').alias('vineyard_ndvi_mean')))

ndvi = (ndvi
    .join(vineyard_mean, on=['year', 'week'], how='left')
    .withColumn('ndvi_anomaly', F.col('ndvi_smooth_mean') - F.col('vineyard_ndvi_mean')))

feature_wks = list(range(1, 36))
target_wks  = list(range(36, 44))
all_wks     = list(range(1, 53))

# Pivot to wide — Spark names columns as {pivot_value}_{agg_alias}
print('Pivoting NDVI features to wide…')
ndvi_wide = (ndvi
    .groupBy('tile_id', 'year')
    .pivot('week', all_wks)
    .agg(
        F.first('ndvi_smooth_mean').alias('ndvi_smooth_mean'),
        F.first('ndvi_smooth_std').alias('ndvi_smooth_std'),
        F.first('ndvi_smooth_slope').alias('ndvi_smooth_slope'),
        F.first('ndvi_anomaly').alias('ndvi_anomaly'),
    ))

# ndvi_anomaly_harvest: null-safe mean of harvest-window anomaly (weeks 36–43)
target_anom_cols = [f'{w}_ndvi_anomaly' for w in target_wks]
anom_sum   = reduce(lambda a, b: a + b, [F.coalesce(F.col(c), F.lit(0.0)) for c in target_anom_cols])
anom_count = reduce(lambda a, b: a + b, [F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in target_anom_cols])
ndvi_wide = ndvi_wide.withColumn(
    'ndvi_anomaly_harvest',
    F.when(anom_count > 0, anom_sum / anom_count).otherwise(F.lit(None).cast('double'))
)

print(f'NDVI wide: {ndvi_wide.count():,} rows × {len(ndvi_wide.columns)} cols')

NDVI weekly rows: 17,247,494
+-------+----+----+----------------+---------------+-----------------+
|tile_id|year|week|ndvi_smooth_mean|ndvi_smooth_std|ndvi_smooth_slope|
+-------+----+----+----------------+---------------+-----------------+
|    150|2016|   1|      0.30410871|            0.0|              0.0|
|    150|2016|   2|      0.30410871|            0.0|              0.0|
|    150|2016|   3|      0.30410871|            0.0|              0.0|
+-------+----+----+----------------+---------------+-----------------+
only showing top 3 rows
Pivoting NDVI features to wide…


NDVI wide: 329,780 rows × 211 cols


In [4]:
# Replicates v1's weather aggregation (06_assemble_data.ipynb)
weather = pd.read_csv(WEATHER)
weather['date'] = pd.to_datetime(weather['date'])
weather['year'] = weather['date'].dt.year
weather['week'] = weather['date'].dt.isocalendar().week.astype(int)
weather['gdd']  = (weather['tmean'] - 10).clip(lower=0)

# Filter to growing-season weeks 28–43 (matches v1 exactly)
clim_wks = weather[(weather['week'] >= 28) & (weather['week'] <= 43)].copy()

# Weekly aggregation — v1 agg_funcs
weekly_long = (
    clim_wks
    .groupby(['year', 'week'])
    .agg({'ppt': 'sum', 'tmax': 'max', 'tmin': 'min', 'tmean': 'mean',
          'vpdmax': 'max', 'vpdmin': 'min', 'gdd': 'sum'})
    .reset_index()
)

# Pivot to one row per year, columns per week (same structure as v1)
clim_wide = pd.DataFrame({'year': sorted(clim_wks['year'].unique())})
for col, alias in [
    ('ppt',    'ppt'),
    ('tmax',   'tmax'),
    ('tmin',   'tmin'),
    ('tmean',  'tmean'),
    ('vpdmax', 'vpdmax'),
    ('vpdmin', 'vpdmin'),
    ('gdd',    'cumulative_gdd'),   # v1 names this cumulative_gdd
]:
    piv = weekly_long.pivot(index='year', columns='week', values=col).reset_index()
    piv.columns = ['year'] + [f'{alias}_{w}' for w in piv.columns[1:]]
    clim_wide = clim_wide.merge(piv, on='year', how='left')

# Derived climate features — exact v1 formulas and naming
# Note: stress_index has no underscore before week number (matches v1's f-string)
for i in range(28, 44):
    clim_wide[f'water_availability_{i}'] = clim_wide[f'ppt_{i}'] / (1 + clim_wide[f'cumulative_gdd_{i}'])
    clim_wide[f'diurnal_temp_range_{i}'] = clim_wide[f'tmax_{i}'] / clim_wide[f'tmin_{i}']
    clim_wide[f'stress_index{i}']        = clim_wide[f'vpdmax_{i}'] / (clim_wide[f'ppt_{i}'] + 0.1)

clim_sp = spark.createDataFrame(clim_wide)
print(f'Climate wide: {clim_sp.count()} rows × {len(clim_sp.columns)} cols')
print('Sample climate cols:', clim_sp.columns[:8])

Climate wide: 10 rows × 161 cols
Sample climate cols: ['year', 'ppt_28', 'ppt_29', 'ppt_30', 'ppt_31', 'ppt_32', 'ppt_33', 'ppt_34']


In [5]:
# Join climate (vineyard-wide per year — broadcast)
wide = ndvi_wide.join(F.broadcast(clim_sp), on='year', how='left')
print(f'After climate join: {wide.count():,} rows x {len(wide.columns)} cols')

# Terrain: derive total_relief (absent from v2 CSV) then add v1 derived cols
terrain_pd = pd.read_csv(TERRAIN)
terrain_sp = (
    spark.createDataFrame(terrain_pd)
    .withColumn('total_relief',     F.col('elev_max') - F.col('elev_min'))
    .withColumn('local_relief',      F.col('elev_mean') - F.col('elev_min'))
    .withColumn('total_relief_log',  F.log1p(F.col('total_relief')))
)
terrain_sp = terrain_sp.cache()
print(f'Terrain: {terrain_sp.count():,} rows x {len(terrain_sp.columns)} cols')

wide = wide.join(F.broadcast(terrain_sp), on='tile_id', how='left')
print(f'After terrain join: {wide.count():,} rows x {len(wide.columns)} cols')

# Soil
soil_pd = pd.read_csv(SOIL)
soil_sp = spark.createDataFrame(soil_pd).cache()
print(f'Soil: {soil_sp.count():,} rows x {len(soil_sp.columns)} cols')

wide = wide.join(F.broadcast(soil_sp), on='tile_id', how='left')
print(f'Assembled: {wide.count():,} rows x {len(wide.columns)} cols')

After climate join: 329,780 rows x 371 cols


26/05/19 23:02:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Terrain: 32,978 rows x 26 cols
After terrain join: 329,780 rows x 396 cols
Soil: 32,978 rows x 24 cols
Assembled: 329,780 rows x 419 cols


In [6]:
wide.write.mode('overwrite').parquet(OUT_DIR)
print(f'Saved → {OUT_DIR}')

Saved → ../data/spark/assembled_v2


In [7]:
check = spark.read.parquet(OUT_DIR)
print(f'Rows: {check.count():,}  Cols: {len(check.columns)}')

spot_cols = [
    'ndvi_anomaly_harvest',
    '25_ndvi_smooth_mean', '25_ndvi_smooth_slope', '25_ndvi_smooth_std',
    'ppt_28', 'cumulative_gdd_28', 'water_availability_28', 'stress_index28',
    'elev_mean', 'local_relief', 'total_relief_log',
]
present = [c for c in spot_cols if c in check.columns]
missing = [c for c in spot_cols if c not in check.columns]
if missing:
    print(f'WARNING — expected columns not found: {missing}')

print('\nNull counts:')
check.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in present]).show()

check.select(
    'tile_id', 'year', 'ndvi_anomaly_harvest',
    '25_ndvi_smooth_mean', '25_ndvi_smooth_slope',
    'ppt_28', 'stress_index28',
    'elev_mean', 'local_relief',
).show(5)

Rows: 329,780  Cols: 419

Null counts:
+--------------------+-------------------+--------------------+------------------+------+-----------------+---------------------+--------------+---------+------------+----------------+
|ndvi_anomaly_harvest|25_ndvi_smooth_mean|25_ndvi_smooth_slope|25_ndvi_smooth_std|ppt_28|cumulative_gdd_28|water_availability_28|stress_index28|elev_mean|local_relief|total_relief_log|
+--------------------+-------------------+--------------------+------------------+------+-----------------+---------------------+--------------+---------+------------+----------------+
|                   0|                  0|                   0|                 0|     0|                0|                    0|             0|        0|           0|               0|
+--------------------+-------------------+--------------------+------------------+------+-----------------+---------------------+--------------+---------+------------+----------------+

+-------+----+--------------------+